In [1]:
import requests
import pandas as pd

Hava verilerini indirme

In [2]:
# 5 yıllık tarih aralığı
start_date = "2021-01-01"
end_date = "2025-12-31"

# şehir koordinatları
cities = {
    "istanbul": (41.01, 28.97),
    "izmir": (38.42, 27.14),
    "ankara": (39.93, 32.85),
    "antalya": (36.89, 30.70),
    "samsun": (41.29, 36.33),
    "gaziantep": (37.07, 37.38),
    "erzurum": (39.90, 41.27)
}

all_data = []

for city, (lat, lon) in cities.items():

    url = "https://archive-api.open-meteo.com/v1/archive"

    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": [
            "temperature_2m",
            "relative_humidity_2m",
            "apparent_temperature",
            "cloud_cover",
            "wind_speed_10m",
            "precipitation"
        ],
        "timezone": "auto"
    }

    response = requests.get(url, params=params)
    data = response.json()

    df = pd.DataFrame(data["hourly"])
    df["city"] = city

    all_data.append(df)

weather_df = pd.concat(all_data)

weather_df.to_csv("turkey_weather_hourly.csv", index=False)

print("Veri başarıyla indirildi")

Veri başarıyla indirildi


**Verileri birleştirme**

In [3]:
electricity = pd.read_csv("C:/Users/kerem/Desktop/elektrikveri/birleşik_veri.csv")
electricity["Datetime"] = pd.to_datetime(electricity["Datetime"])
electricity = electricity.sort_values("Datetime")

weather = pd.read_csv("turkey_weather_hourly.csv")
weather["time"] = pd.to_datetime(weather["time"])

 HAVA VERİSİNİ PIVOTLA

In [4]:
weather_pivot = weather.pivot(
    index="time",
    columns="city",
    values=[
        "temperature_2m",
        "relative_humidity_2m",
        "apparent_temperature",
        "cloud_cover",
        "wind_speed_10m",
        "precipitation"
    ]
)

weather_pivot.columns = [
    f"{var}_{city}" for var, city in weather_pivot.columns
]

weather_pivot = weather_pivot.reset_index()
weather_pivot.rename(columns={"time": "Datetime"}, inplace=True)

ŞEHİR AĞIRLIKLARI

In [5]:
weights = {
    "istanbul": 0.30,
    "ankara": 0.15,
    "izmir": 0.13,
    "antalya": 0.12,
    "gaziantep": 0.10,
    "samsun": 0.10,
    "erzurum": 0.10
}

weather_vars = [
    "temperature_2m",
    "relative_humidity_2m",
    "apparent_temperature",
    "cloud_cover",
    "wind_speed_10m",
    "precipitation"
]

# -------------------------------------------------
# 5. AĞIRLIKLI ORTALAMA HAVA DEĞİŞKENLERİ
# -------------------------------------------------

for var in weather_vars:
    weather_pivot[f"{var}_weighted"] = sum(
        weather_pivot[f"{var}_{city}"] * weight
        for city, weight in weights.items()
    )

In [6]:
dataset = pd.merge(
    electricity,
    weather_pivot,
    on="Datetime",
    how="left"
)

In [7]:
print(weather.head())

                 time  temperature_2m  relative_humidity_2m  \
0 2021-01-01 00:00:00            12.6                    75   
1 2021-01-01 01:00:00            13.2                    69   
2 2021-01-01 02:00:00            12.8                    73   
3 2021-01-01 03:00:00            12.6                    75   
4 2021-01-01 04:00:00            12.3                    77   

   apparent_temperature  cloud_cover  wind_speed_10m  precipitation      city  
0                   9.9           93            15.2            0.0  istanbul  
1                   9.6           44            20.7            0.0  istanbul  
2                   9.4           45            20.2            0.0  istanbul  
3                   9.2           54            19.8            0.0  istanbul  
4                   9.3           45            17.7            0.1  istanbul  


In [8]:
print(weather_pivot.head())

             Datetime  temperature_2m_ankara  temperature_2m_antalya  \
0 2021-01-01 00:00:00                    2.5                    11.7   
1 2021-01-01 01:00:00                    1.7                    11.3   
2 2021-01-01 02:00:00                    1.3                    10.8   
3 2021-01-01 03:00:00                    1.7                    10.9   
4 2021-01-01 04:00:00                    1.3                    10.9   

   temperature_2m_erzurum  temperature_2m_gaziantep  temperature_2m_istanbul  \
0                    -6.8                       6.0                     12.6   
1                    -6.5                       6.3                     13.2   
2                    -6.4                       6.4                     12.8   
3                    -6.4                       6.9                     12.6   
4                    -6.6                       6.5                     12.3   

   temperature_2m_izmir  temperature_2m_samsun  relative_humidity_2m_ankara  \
0      

In [9]:
dataset.to_csv("electricity_weather_dataset.csv", index=False)

In [10]:
dataset.columns

Index(['Datetime', 'Üretim Miktarı (MWh)', 'Tüketim Miktarı(MWh)',
       'Ortalama Fiyat (TL/MWh)', 'temperature_2m_ankara',
       'temperature_2m_antalya', 'temperature_2m_erzurum',
       'temperature_2m_gaziantep', 'temperature_2m_istanbul',
       'temperature_2m_izmir', 'temperature_2m_samsun',
       'relative_humidity_2m_ankara', 'relative_humidity_2m_antalya',
       'relative_humidity_2m_erzurum', 'relative_humidity_2m_gaziantep',
       'relative_humidity_2m_istanbul', 'relative_humidity_2m_izmir',
       'relative_humidity_2m_samsun', 'apparent_temperature_ankara',
       'apparent_temperature_antalya', 'apparent_temperature_erzurum',
       'apparent_temperature_gaziantep', 'apparent_temperature_istanbul',
       'apparent_temperature_izmir', 'apparent_temperature_samsun',
       'cloud_cover_ankara', 'cloud_cover_antalya', 'cloud_cover_erzurum',
       'cloud_cover_gaziantep', 'cloud_cover_istanbul', 'cloud_cover_izmir',
       'cloud_cover_samsun', 'wind_speed_10m_ankar